# LangGraph Dual Model Agent: Llama-3.2-1B + Qwen2.5-1.5B

This notebook demonstrates a **dual-model LangGraph agent** built with the **Message API**, using **Llama-3.2-1B-Instruct** and **Qwen2.5-1.5B-Instruct**.

## Features

- **Dual Model Support**: Run Llama and Qwen simultaneously with intelligent routing
- **Unified Conversation History**: Both models share the same history - they can reference each other's responses!
- **Smart Routing**: 
  - Say "Hey Llama" → only Llama responds
  - Say "Hey Qwen" → only Qwen responds  
  - Say neither → BOTH models respond
- **Message API**: Uses SystemMessage, HumanMessage, and AIMessage for proper conversation handling
- **Bracket Prefixes**: All messages tagged with `[Human]`, `[Llama]`, or `[Qwen]`
- **History Transformation**: Each model sees their own responses as "assistant" and others as "user"
- **Automatic History Management**: LangGraph's `add_messages` reducer maintains conversation history
- **Device Detection**: Automatically uses CUDA (NVIDIA GPU), MPS (Apple Silicon), or CPU

## How Multi-Participant Conversations Work

Instead of a simple user-assistant dialog, this agent supports three participants: Human, Llama, and Qwen.

### Key Innovation: Unified But Transformed History

**Storage**: All messages stored together in one unified list:
```python
[
  HumanMessage(content="[Human] What's 2+2?"),
  AIMessage(content="[Llama] It's 4", additional_kwargs={"model": "llama"}),
  AIMessage(content="[Qwen] Yes, 4", additional_kwargs={"model": "qwen"})
]
```

**Transformation**: Each model sees history differently:
- **Llama** sees its own responses as `role="assistant"`, Qwen's as `role="user"`
- **Qwen** sees its own responses as `role="assistant"`, Llama's as `role="user"`
- **Human** messages always appear as `role="user"` to both

### Example Conversation

```
Human: "Hey Llama, my favorite number is 42"
Llama: "[Llama] I'll remember that your favorite number is 42!"

In [37]:
%pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [38]:
%pip install torch transformers langchain langchain-huggingface langgraph grandalf langgraph-checkpoint-sqlite

In [39]:
# Import necessary libraries
import sys
import time
import warnings
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from datetime import datetime
from getpass import getpass
import operator

# Import LangChain message types for Message API
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

# Import checkpointing support for crash recovery
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# The model's generation_config.json sets max_length=20; the pipeline overrides
# it with max_new_tokens=256.  Suppress the redundant warning.
warnings.filterwarnings(
    "ignore",
    message="Both `max_new_tokens`.*and `max_length`"
)

In [40]:
# Determine the best available device for inference
# Priority: CUDA (NVIDIA GPU) > MPS (Apple Silicon) > CPU
def get_device():
    """
    Detect and return the best available compute device.
    Returns 'cuda' for NVIDIA GPUs, 'mps' for Apple Silicon, or 'cpu' as fallback.
    """
    if torch.cuda.is_available():
        print("Using CUDA (NVIDIA GPU) for inference")
        return "cuda"
    elif torch.backends.mps.is_available():
        print("Using MPS (Apple Silicon) for inference")
        return "mps"
    else:
        print("Using CPU for inference")
        return "cpu"

In [41]:
# =============================================================================
# STATE DEFINITION
# =============================================================================
# The state is a TypedDict that flows through all nodes in the graph.
# LangGraph's add_messages reducer automatically manages conversation history.

class AgentState(TypedDict):
    """
    State object that flows through the LangGraph nodes.
    
    Fields:
    - messages: List of conversation messages (SystemMessage, HumanMessage, AIMessage)
        Managed automatically by the add_messages reducer which:
        - Appends new messages to existing list
        - Preserves message types and order
        - Maintains full conversation history
        - UNIFIED across both Llama and Qwen - models can see each other's responses
    - should_exit: Boolean flag indicating if user wants to quit
    - skip_llm: Boolean flag to skip LLM call (for empty input)
    - active_models: List of model names to invoke (["llama"], ["qwen"], or ["llama", "qwen"])
    - last_displayed_index: Index tracking which messages have been displayed to user
    
    State Flow:
        START → get_user_input → call_llm_loop → print_response → get_user_input
                     ↑_______________________________________________|
                     
        Exit: get_user_input → END (when should_exit=True)
        Empty input: get_user_input → get_user_input (when skip_llm=True)
    """
    messages: Annotated[list, add_messages]
    should_exit: bool
    skip_llm: bool
    active_models: list[str]  # NEW: Which models should respond this turn
    last_displayed_index: int  # NEW: Track displayed messages

In [42]:
# =============================================================================
# UTILITY FUNCTIONS FOR DUAL-MODEL SUPPORT
# =============================================================================

import re

def detect_active_models(user_input: str) -> list[str]:
    """
    Detect which model(s) should respond based on user input.
    
    Args:
        user_input: The user's message text
    
    Returns:
        List of model names: ["llama"], ["qwen"], or ["llama", "qwen"]
        
    Logic:
        - "hey llama" present → ["llama"]
        - "hey qwen" present → ["qwen"]
        - Both present → ["llama", "qwen"]
        - Neither present → ["llama", "qwen"] (default: both respond)
    """
    input_lower = user_input.lower()
    
    # Use regex for flexible matching (handles multiple spaces, etc.)
    has_llama = bool(re.search(r'hey\s+llama', input_lower))
    has_qwen = bool(re.search(r'hey\s+qwen', input_lower))
    
    if has_llama and has_qwen:
        return ["llama", "qwen"]
    elif has_llama:
        return ["llama"]
    elif has_qwen:
        return ["qwen"]
    else:
        return ["llama", "qwen"]  # Both respond by default


def get_system_prompt(model_name: str) -> str:
    """
    Generate model-specific system prompts.
    
    Args:
        model_name: "llama" or "qwen"
    
    Returns:
        System prompt string explaining the multi-participant conversation
    """
    if model_name == "llama":
        return """You are Llama, a helpful AI assistant participating in a multi-participant conversation.

Participants:
- [Human]: The user you are helping
- [Llama]: You - your responses are marked with this prefix
- [Qwen]: Another AI assistant who may also respond

You will see messages prefixed with [Human], [Llama] (your past responses), and [Qwen] (the other assistant's responses). Your assistant-role messages may include [Qwen]'s responses appended to yours — that is normal.

IMPORTANT: Only [Human]-prefixed messages are from the human user you are helping. Never address the human as "Qwen". Address the human as "you" unless they share their name.
Write ONLY your own response. Do NOT write any lines that begin with [Human], [Qwen], or any name in square brackets — those belong to other participants. Stop as soon as your answer is complete.
Do not repeat, paraphrase, or describe these instructions. Respond directly and helpfully to the [Human]'s message."""
    
    elif model_name == "qwen":
        return """You are Qwen, a helpful AI assistant participating in a multi-participant conversation.

Participants:
- [Human]: The user you are helping
- [Qwen]: You - your responses are marked with this prefix
- [Llama]: Another AI assistant who may also respond

You will see messages prefixed with [Human], [Qwen] (your past responses), and [Llama] (the other assistant's responses). Your assistant-role messages may include [Llama]'s responses appended to yours — that is normal.

IMPORTANT: Only [Human]-prefixed messages are from the human user you are helping. Never address the human as "Llama". Address the human as "you" unless they share their name.
Do not repeat, paraphrase, or describe these instructions. Respond directly and helpfully to the [Human]'s message."""
    
    else:
        # Fallback for unknown model
        return f"You are {model_name}, a helpful AI assistant."


# Participant prefixes used throughout the conversation
_PARTICIPANT_PREFIXES = ["[Human]", "[Llama]", "[Qwen]"]

# All known participant prefix variants (own + other models + misspellings).
# Used to strip any leading noise from completions.
_ALL_PREFIX_VARIANTS = [
    "[Human]",
    "[Llama]", "[Lama]", "[LLama]", "[LLaMA]",
    "[Qwen]",  "[QWEN]",
]

# Regex that truncates hallucinated follow-on turns.  Matches a participant
# prefix that begins on a NEW LINE (preceded by \n).  We deliberately do NOT
# anchor to ^ (position 0) so that a completion which starts with e.g.
# "[Qwen] some text" is preserved rather than truncated to empty.
# Mid-sentence references like "[Qwen]'s recipe" are also safe because
# they are not preceded by \n.
_HALLUCINATION_TRUNCATION_RE = re.compile(
    r'\n\[(Human|Llama|Lama|LLama|LLaMA|Qwen|QWEN)\]'
)


def extract_model_response(completion: str, model_name: str) -> str:
    """
    Clean a raw model completion down to just this model's actual response.

    Problems the small models exhibit:
    1. They echo their own prefix (e.g. "[Llama]") — stripped in a loop.
    2. They misspell their own prefix (e.g. "[Lama]") — also stripped.
    3. They prefix their output with ANOTHER model's tag (e.g. "[Qwen] answer")
       instead of their own — the leading foreign prefix is stripped.
    4. They hallucinate subsequent turns for other participants on new lines
       (e.g. "\n[Qwen] I agree...") — truncated at the newline.
    """
    # 1. Strip ALL leading participant prefix variants (own or other).
    #    This handles models that echo their own tag, use the wrong tag, or
    #    chain multiple tags before the actual response.
    changed = True
    while changed:
        changed = False
        for variant in _ALL_PREFIX_VARIANTS:
            if completion.startswith(variant):
                completion = completion[len(variant):].lstrip()
                changed = True
                break

    # 2. Truncate at the first participant prefix that starts a NEW LINE —
    #    those are hallucinated follow-on turns, not part of this response.
    #    We never truncate at position 0; content that remains after step 1
    #    is always kept up to the first \n[Participant] boundary.
    m = _HALLUCINATION_TRUNCATION_RE.search(completion)
    truncate_at = m.start() if m else len(completion)

    return completion[:truncate_at].strip()


def transform_messages_for_model(messages: list, target_model: str) -> list[dict]:
    """
    Transform conversation history for a specific model.
    
    Each model sees:
    - Their own responses as role="assistant"
    - Human messages as role="user"
    - Other model's responses appended to the preceding assistant message
      (rather than as a separate role="user" entry, which would cause the
      other model's name to bleed into the next human-role message and
      confuse the model about the human's identity)
    
    A final merge pass combines any consecutive messages that share the same
    role as a safety net.
    
    Args:
        messages: Full conversation history (LangChain message objects)
        target_model: "llama" or "qwen" - which model will receive these messages
    
    Returns:
        List of dicts in format for apply_chat_template:
        [{"role": "system"/"user"/"assistant", "content": "..."}]
    """
    formatted_messages = []
    
    # Always start with model-specific system prompt
    formatted_messages.append({
        "role": "system",
        "content": get_system_prompt(target_model)
    })
    
    # Transform each message based on its type and source
    for msg in messages:
        if isinstance(msg, SystemMessage):
            # Skip original system message - we already added model-specific one
            continue
        
        elif isinstance(msg, HumanMessage):
            # Human messages are always role="user"
            formatted_messages.append({
                "role": "user",
                "content": msg.content
            })
        
        elif isinstance(msg, AIMessage):
            # Check which model produced this message
            source_model = msg.additional_kwargs.get("model", "unknown")
            
            if source_model == target_model:
                # This model's own past responses → role="assistant"
                formatted_messages.append({
                    "role": "assistant",
                    "content": msg.content
                })
            else:
                # Other model's responses — place them so this model knows what
                # the other model said without mistaking it for its own prior answer.
                #
                # Preferred: append to the preceding assistant message (Qwen's own
                # previous turn), keeping the other model's content visible while
                # the role structure stays clean.
                #
                # Fallback (no preceding assistant message yet, e.g. Llama just
                # answered in the *same* turn as Qwen's first response): append to
                # the preceding user message.  Creating a new assistant message here
                # would make the target model think it already answered, causing it
                # to produce a follow-up ("I'm glad I could help…") instead of its
                # own independent answer.
                if formatted_messages and formatted_messages[-1]["role"] == "assistant":
                    formatted_messages[-1]["content"] += f"\n\n{msg.content}"
                elif formatted_messages and formatted_messages[-1]["role"] == "user":
                    formatted_messages[-1]["content"] += f"\n\n{msg.content}"
                else:
                    # No prior message at all — create an assistant entry as a
                    # last resort so the list is never empty.
                    formatted_messages.append({
                        "role": "assistant",
                        "content": msg.content
                    })
    
    # Merge consecutive messages with the same role (safety net).
    # The other-model appending above handles the common case, but this
    # guards against any edge cases that still produce consecutive same-role
    # messages.
    merged_messages = []
    for msg in formatted_messages:
        if merged_messages and merged_messages[-1]["role"] == msg["role"]:
            merged_messages[-1]["content"] += "\n\n" + msg["content"]
        else:
            merged_messages.append(msg)
    
    return merged_messages

In [43]:
# =============================================================================
# DUAL LLM CREATION
# =============================================================================

def create_dual_llms():
    """
    Load both Llama-3.2-1B-Instruct and Qwen2.5-1.5B-Instruct models.
    
    Memory considerations:
    - Llama-3.2-1B-Instruct: ~2-3GB in fp16
    - Qwen2.5-1.5B-Instruct: ~3-4GB in fp16
    - Total: ~5-7GB GPU memory required
    
    Returns:
        tuple: (llama_llm, llama_tokenizer, qwen_llm, qwen_tokenizer)
    """
    # Get the optimal device for inference
    device = get_device()

    # Prompt for HuggingFace token (required for gated models like Llama)
    print("\nPlease enter your HuggingFace token:")
    print("(Get your token from: https://huggingface.co/settings/tokens)")
    hf_token = getpass("HF Token: ")

    # ==========================================================================
    # LOAD LLAMA
    # ==========================================================================
    print("\n" + "=" * 80)
    print("Loading Llama-3.2-1B-Instruct...")
    print("=" * 80)

    llama_tokenizer = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-1B-Instruct",
        token=hf_token
    )
    
    llama_model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.2-1B-Instruct",
        token=hf_token,
        torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    
    if device == "mps":
        llama_model = llama_model.to(device)
    
    llama_pipe = pipeline(
        "text-generation",
        model=llama_model,
        tokenizer=llama_tokenizer,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        pad_token_id=llama_tokenizer.eos_token_id,
    )
    
    llama_llm = HuggingFacePipeline(pipeline=llama_pipe)
    print("Llama loaded successfully!")

    # ==========================================================================
    # LOAD QWEN
    # ==========================================================================
    print("\n" + "=" * 80)
    print("Loading Qwen2.5-1.5B-Instruct...")
    print("=" * 80)

    qwen_tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct"
    )
    
    qwen_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    
    if device == "mps":
        qwen_model = qwen_model.to(device)
    
    qwen_pipe = pipeline(
        "text-generation",
        model=qwen_model,
        tokenizer=qwen_tokenizer,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        pad_token_id=qwen_tokenizer.eos_token_id,
    )
    
    qwen_llm = HuggingFacePipeline(pipeline=qwen_pipe)
    print("Qwen loaded successfully!")

    print("\n" + "=" * 80)
    print("Both models loaded successfully!")
    print("=" * 80)
    
    return llama_llm, llama_tokenizer, qwen_llm, qwen_tokenizer

In [44]:
# =============================================================================
# GRAPH CREATION FUNCTION - DUAL MODEL MESSAGE API APPROACH
# =============================================================================

def create_graph(llama_llm, llama_tokenizer, qwen_llm, qwen_tokenizer, debug=False):
    """
    Create the LangGraph state graph using Message API with dual-model support.
    Manually applies chat template for proper message formatting.
    
    Args:
        llama_llm: Llama language model
        llama_tokenizer: Llama tokenizer for chat template
        qwen_llm: Qwen language model
        qwen_tokenizer: Qwen tokenizer for chat template
        debug: Enable debug output (default: False)
    """

    # =========================================================================
    # NODES
    # =========================================================================
    def get_user_input(state: AgentState) -> dict:
        """Get user input, detect active models, and create HumanMessage with [Human] prefix."""
        print("\n" + "=" * 50)
        print("Enter your text (or 'quit' to exit):")
        print("=" * 50)

        print("\n> ", end="")
        sys.stdout.flush()
        user_input = input()

        if not user_input.strip():
            print("\n[Empty input - please enter some text]")
            # Set a flag to skip LLM call
            return {"messages": [], "should_exit": False, "skip_llm": True, "active_models": []}

        print(f"\nYou: {user_input}")

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            return {"messages": [], "should_exit": True, "skip_llm": True, "active_models": []}

        # Detect which models should respond
        active_models = detect_active_models(user_input)
        
        if debug:
            print(f"\n[DEBUG get_user_input] Active models: {active_models}")

        # Add [Human] prefix to user message
        prefixed_message = f"[Human] {user_input}"
        
        # Create HumanMessage - add_messages reducer will append it
        return {
            "messages": [HumanMessage(content=prefixed_message)],
            "should_exit": False,
            "skip_llm": False,
            "active_models": active_models
        }

    def call_llm_loop(state: AgentState) -> dict:
        """
        Invoke each active model against the same frozen snapshot of history
        from before this turn.  Each model's response is returned as an
        AIMessage and committed to shared state after the node returns, where
        it becomes visible to all models on the next turn.

        Both models receive the same frozen snapshot of history from before
        this turn.  Each model's response is committed to state after the node
        returns and becomes visible to all models on the next turn.
        """
        active_models = state["active_models"]
        messages_snapshot = list(state["messages"])  # frozen — same for every model
        new_messages = []
        
        if debug:
            print(f"\n[DEBUG call_llm_loop] Processing {len(active_models)} model(s)")
            print(f"[DEBUG call_llm_loop] Current message count: {len(messages_snapshot)}")
        
        # Process each model in order (Llama first, then Qwen)
        for model_name in active_models:
            if debug:
                print(f"\n[DEBUG call_llm_loop] Calling {model_name}...")
            
            # Get appropriate LLM and tokenizer
            if model_name == "llama":
                llm, tokenizer = llama_llm, llama_tokenizer
            else:  # qwen
                llm, tokenizer = qwen_llm, qwen_tokenizer
            
            # Transform the frozen snapshot for this model — every model in
            # this turn sees the same pre-turn history.
            formatted_messages = transform_messages_for_model(messages_snapshot, model_name)
            
            if debug:
                print(f"[DEBUG call_llm_loop] Transformed {len(formatted_messages)} messages for {model_name}")
                print(f"[DEBUG call_llm_loop] Last 2 messages:")
                for msg in formatted_messages[-2:]:
                    preview = msg["content"][:50] + "..." if len(msg["content"]) > 50 else msg["content"]
                    print(f"  {msg['role']}: {preview}")
            
            # Direct model generation — tokenize once, slice by token count.
            # The old approach (apply_chat_template(tokenize=False) + llm.invoke(prompt)
            # + full_response[len(prompt):]) breaks for Llama because multi-byte
            # special tokens don't survive an encode→decode roundtrip unchanged,
            # so the character-offset slice lands in the wrong place and yields
            # an empty completion.  Slicing by token index is exact and
            # encoding-agnostic.
            raw_model = llm.pipeline.model
            device = next(raw_model.parameters()).device

            encoded = tokenizer.apply_chat_template(
                formatted_messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            # apply_chat_template returns a bare Tensor in older transformers but
            # a BatchEncoding (dict-like) in newer versions; extract accordingly.
            # Also extract the attention_mask when available, otherwise build a
            # ones tensor so generate() doesn't emit the pad==eos warning.
            if isinstance(encoded, torch.Tensor):
                input_ids = encoded.to(device)
                attention_mask = torch.ones_like(input_ids)
            else:
                input_ids = encoded["input_ids"].to(device)
                if "attention_mask" in encoded:
                    attention_mask = encoded["attention_mask"].to(device)
                else:
                    attention_mask = torch.ones_like(input_ids)

            if debug:
                print(f"[DEBUG call_llm_loop] Prompt length: {input_ids.shape[1]} tokens")

            print(f"\n[Processing {model_name.upper()}...]")
            sys.stdout.flush()

            with torch.no_grad():
                output_ids = raw_model.generate(
                    input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=256,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.95,
                    pad_token_id=tokenizer.eos_token_id,
                )

            # Decode only the newly generated tokens (not the prompt)
            completion = tokenizer.decode(
                output_ids[0][input_ids.shape[1]:],
                skip_special_tokens=True,
            )
            
            # Strip echoed self-prefix and truncate any hallucinated
            # other-participant turns before storing.
            completion = extract_model_response(completion, model_name)
            
            if debug:
                comp_preview = completion[:100] + "..." if len(completion) > 100 else completion
                print(f"[DEBUG call_llm_loop] Extracted completion: {comp_preview}")
            
            # Add model prefix to response
            prefixed_response = f"[{model_name.capitalize()}] {completion}"
            
            # Create AIMessage with metadata
            ai_message = AIMessage(
                content=prefixed_response,
                additional_kwargs={"model": model_name}
            )
            
            new_messages.append(ai_message)
            # No append to messages_snapshot — each model sees the same
            # pre-turn history.  Responses are committed to shared state
            # after this node returns via the add_messages reducer.
        
        if debug:
            print(f"\n[DEBUG call_llm_loop] Returning {len(new_messages)} new message(s)")
        
        return {
            "messages": new_messages,
            "skip_llm": False
        }

    def print_response(state: AgentState) -> dict:
        """Print only NEW AIMessages that haven't been displayed yet."""
        messages = state["messages"]
        last_displayed = state.get("last_displayed_index", 0)
        
        if debug:
            print(f"\n[DEBUG print_response] Total messages: {len(messages)}")
            print(f"[DEBUG print_response] Last displayed index: {last_displayed}")
        
        # Get messages after last_displayed_index
        new_messages = messages[last_displayed:]
        
        # Print only AIMessages
        for msg in new_messages:
            if isinstance(msg, AIMessage):
                model_name = msg.additional_kwargs.get("model", "unknown")
                
                # Strip the leading [ModelName] prefix for display — it is
                # already shown in the header and is only kept in msg.content
                # for the conversation history.
                display_text = msg.content
                display_prefix = f"[{model_name.capitalize()}]"
                if display_text.startswith(display_prefix):
                    display_text = display_text[len(display_prefix):].lstrip()
                
                print("\n" + "=" * 80)
                print(f"[{model_name.upper()}]")
                print("-" * 80)
                print(display_text)
                print("=" * 80)
                
                sys.stdout.flush()
        
        # Small delay for readability
        if any(isinstance(msg, AIMessage) for msg in new_messages):
            time.sleep(0.1)
            print()
            sys.stdout.flush()
        
        # Update last_displayed_index to current message count
        return {
            "skip_llm": False,
            "last_displayed_index": len(messages)
        }

    # =========================================================================
    # CONDITIONAL ROUTING
    # =========================================================================
    def route_after_input(state: AgentState) -> str:
        """Route based on whether user wants to exit or skip LLM."""
        if state.get("should_exit", False):
            return END
        elif state.get("skip_llm", False):
            # Skip LLM and go directly back to input
            return "get_user_input"
        else:
            return "call_llm_loop"

    # =========================================================================
    # GRAPH CONSTRUCTION
    # =========================================================================
    graph_builder = StateGraph(AgentState)

    # Add nodes
    graph_builder.add_node("get_user_input", get_user_input)
    graph_builder.add_node("call_llm_loop", call_llm_loop)
    graph_builder.add_node("print_response", print_response)

    # Add edges
    graph_builder.add_edge(START, "get_user_input")
    
    # Conditional routing after user input
    graph_builder.add_conditional_edges(
        "get_user_input",
        route_after_input,
        {
            "call_llm_loop": "call_llm_loop",
            "get_user_input": "get_user_input",  # Loop back for empty input
            END: END
        }
    )
    
    # Linear flow: call_llm_loop → print_response → loop back to get_user_input
    graph_builder.add_edge("call_llm_loop", "print_response")
    graph_builder.add_edge("print_response", "get_user_input")

    # Compile with SqliteSaver for crash recovery
    conn = sqlite3.connect("checkpoint.db", check_same_thread=False)
    checkpointer = SqliteSaver(conn)
    graph = graph_builder.compile(checkpointer=checkpointer)

    if debug:
        print("[DEBUG create_graph] Graph compiled with SqliteSaver checkpointing")

    return graph

In [45]:
# =============================================================================
# GRAPH VISUALIZATION
# =============================================================================

def save_graph_image(graph, filename="lg_graph.png"):
    """
    Generate a Mermaid diagram of the graph and save it as a PNG image.
    Uses the graph's built-in Mermaid export functionality.
    """
    try:
        # Get the Mermaid PNG representation of the graph
        # This requires the 'grandalf' package for rendering
        png_data = graph.get_graph(xray=True).draw_mermaid_png()
        
        # Write the PNG data to file
        with open(filename, "wb") as f:
            f.write(png_data)
        
        print(f"Graph image saved to {filename}")
    except Exception as e:
        print(f"Could not save graph image: {e}")
        print("You may need to install additional dependencies: pip install grandalf")

In [46]:
# =============================================================================
# MAIN FUNCTION - DUAL MODEL MESSAGE API APPROACH
# =============================================================================

# File used to persist the active thread ID across notebook restarts.
# Without this, a "Start fresh" session creates a timestamped thread ID that
# is lost on the next restart, which always fell back to "dual-chat-session".
_THREAD_ID_FILE = "thread_id.txt"

def _load_thread_id() -> str:
    """Return the last-saved thread ID, or the default if the file is absent."""
    try:
        with open(_THREAD_ID_FILE, "r") as f:
            tid = f.read().strip()
            if tid:
                return tid
    except FileNotFoundError:
        pass
    return "dual-chat-session"

def _save_thread_id(thread_id: str) -> None:
    """Persist the active thread ID so the next restart resumes this session."""
    with open(_THREAD_ID_FILE, "w") as f:
        f.write(thread_id)


def main(debug=False):
    """
    Main function that orchestrates the dual-model agent using Message API.
    
    Args:
        debug: Enable debug output (default: False)
    """
    print("=" * 80)
    print("LangGraph Dual Model Agent: Llama-3.2-1B + Qwen2.5-1.5B")
    print("=" * 80)
    print()

    # Load both models
    llama_llm, llama_tokenizer, qwen_llm, qwen_tokenizer = create_dual_llms()

    print("\nCreating LangGraph with dual model support...")
    graph = create_graph(llama_llm, llama_tokenizer, qwen_llm, qwen_tokenizer, debug=debug)
    print("Graph created successfully!")

    print("\nSaving graph visualization...")
    save_graph_image(graph)

    # Load persisted thread ID (resumes the most recent session by default)
    thread_id = _load_thread_id()
    config = {"configurable": {"thread_id": thread_id}}

    # Check for existing checkpoint to enable crash recovery
    state_snapshot = graph.get_state(config)

    print(f"\n" + "=" * 80)
    print("Starting Dual Model Agent")
    print("=" * 80)
    print("\nCRASH RECOVERY ENABLED:")
    print("  - Conversation state saved to checkpoint.db")
    print("  - Kill process (Ctrl+C) and restart - conversation will resume!")
    print("  - To start fresh: delete checkpoint.db")
    print("\nROUTING RULES:")
    print("  - Say 'Hey Llama' → only Llama responds")
    print("  - Say 'Hey Qwen' → only Qwen responds")
    print("  - Otherwise → BOTH models respond")
    print("\nThe conversation history is UNIFIED - models can see each other's responses!")
    print("Type 'quit', 'exit', or 'q' to stop.\n")

    if state_snapshot.next:
        # Incomplete checkpoint exists (crash scenario)
        print("=" * 80)
        print("RESUMING FROM CHECKPOINT")
        print("=" * 80)
        print(f"Found existing conversation with {len(state_snapshot.values.get('messages', []))} messages")
        print(f"Thread ID: {thread_id}")
        print("Resuming from last checkpoint...\n")
        sys.stdout.flush()

        # Resume from checkpoint - pass None to continue from interrupted state
        graph.invoke(None, config=config)
    else:
        if state_snapshot.values.get('messages'):
            msg_count = len(state_snapshot.values['messages'])
            print(f"\nFound previous conversation ({msg_count} messages).")
            print(f"Thread ID: {thread_id}")
            print("  [c] Continue   [f] Start fresh")
            sys.stdout.flush()
            choice = input("Choice: ").strip().lower()

            if choice != 'c':
                # Start fresh — new thread_id so prior history is not in context.
                # Persist the new thread_id so the next restart picks it up.
                thread_id = f"dual-chat-{int(time.time())}"
                config = {"configurable": {"thread_id": thread_id}}
                _save_thread_id(thread_id)
                print(f"\nThread ID: {thread_id}")
                state_snapshot = graph.get_state(config)  # empty for new thread
                print("Starting new conversation...\n")
                sys.stdout.flush()
                initial_state: AgentState = {
                    "messages": [],
                    "should_exit": False,
                    "skip_llm": False,
                    "active_models": [],
                    "last_displayed_index": 0
                }
            else:
                # Continuing — ensure the file reflects this thread_id.
                _save_thread_id(thread_id)
                prior_messages = state_snapshot.values['messages']
                print(f"\nContinuing conversation ({msg_count} messages):\n")

                # Replay existing history so the user can see it
                print("=" * 80)
                print("PREVIOUS CONVERSATION")
                print("=" * 80)
                for msg in prior_messages:
                    if isinstance(msg, HumanMessage):
                        content = msg.content
                        if content.startswith("[Human] "):
                            content = content[len("[Human] "):]
                        print(f"\nYou: {content}")
                    elif isinstance(msg, AIMessage):
                        model_name = msg.additional_kwargs.get("model", "unknown")
                        display_text = msg.content
                        display_prefix = f"[{model_name.capitalize()}]"
                        if display_text.startswith(display_prefix):
                            display_text = display_text[len(display_prefix):].lstrip()
                        print("\n" + "=" * 80)
                        print(f"[{model_name.upper()}]")
                        print("-" * 80)
                        print(display_text)
                        print("=" * 80)
                print("\n" + "=" * 80)
                print("CONTINUING CONVERSATION")
                print("=" * 80 + "\n")
                sys.stdout.flush()  # Flush replay output before graph starts

                initial_state: AgentState = {
                    "messages": [],
                    "should_exit": False,
                    "skip_llm": False,
                    "active_models": [],
                    "last_displayed_index": len(prior_messages)
                }
        else:
            # Fresh start — persist so the next restart returns here.
            _save_thread_id(thread_id)
            print(f"Thread ID: {thread_id}")
            print("Starting new conversation...\n")
            sys.stdout.flush()
            initial_state: AgentState = {
                "messages": [],
                "should_exit": False,
                "skip_llm": False,
                "active_models": [],
                "last_displayed_index": 0
            }

        # Start (or continue) conversation - the checkpointer will merge with existing state
        graph.invoke(initial_state, config=config)

In [47]:
# =============================================================================
# RUN THE AGENT
# =============================================================================
# Set debug=True for detailed debugging output

main()

LangGraph Dual Model Agent: Llama-3.2-1B + Qwen2.5-1.5B

Using CUDA (NVIDIA GPU) for inference

Please enter your HuggingFace token:
(Get your token from: https://huggingface.co/settings/tokens)

Loading Llama-3.2-1B-Instruct...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Llama loaded successfully!

Loading Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen loaded successfully!

Both models loaded successfully!

Creating LangGraph with dual model support...
Graph created successfully!

Saving graph visualization...
Graph image saved to lg_graph.png

Starting Dual Model Agent

CRASH RECOVERY ENABLED:
  - Conversation state saved to checkpoint.db
  - Kill process (Ctrl+C) and restart - conversation will resume!
  - To start fresh: delete checkpoint.db

ROUTING RULES:
  - Say 'Hey Llama' → only Llama responds
  - Say 'Hey Qwen' → only Qwen responds
  - Otherwise → BOTH models respond

The conversation history is UNIFIED - models can see each other's responses!
Type 'quit', 'exit', or 'q' to stop.


Found previous conversation (3 messages).
Thread ID: dual-chat-1771987853
  [c] Continue   [f] Start fresh

Thread ID: dual-chat-1771988405
Starting new conversation...


Enter your text (or 'quit' to exit):

> 
You: I have chicken, peppers, and coconut milk in my fridge. What are some asian inspired recipes that I can make with these ingredi